# 01 · Unity Catalog governance
* PII columns in `silver_customers` are **tagged** (`pii`) so they are discoverable and auditable.
* A masking function + `customers_secure` view expose customers to analysts, the app and Genie
  **without** names/emails/phones unless the reader is in the `valtera_voc_pii_readers` group.
* Gold tables carry no PII by design (customer_id only), so Genie and the app never need raw PII.

In [ ]:
import json, os
for k, v in {"catalog": "daria_krasavina", "schema": "voc_valtera",
             "evidence_path": "/Volumes/daria_krasavina/voc_valtera/evidence"}.items():
    dbutils.widgets.text(k, v)
cat, sch, ev = (dbutils.widgets.get(k) for k in ["catalog", "schema", "evidence_path"])
ns = f"{cat}.{sch}"
log = []

def run(q, tolerate=False):
    try:
        spark.sql(q); log.append({"sql": q.strip()[:160], "ok": True})
    except Exception as e:
        if not tolerate: raise
        log.append({"sql": q.strip()[:160], "ok": False, "error": str(e)[:200]})
print(f"Governance for {ns}: PII tags, masking function, secure view")

Governance for daria_krasavina.voc_valtera: PII tags, masking function, secure view


In [ ]:
for col in ["first_name", "last_name", "email", "phone"]:
    run(f"ALTER TABLE {ns}.silver_customers ALTER COLUMN {col} SET TAGS ('pii' = '{col}')", tolerate=True)
run(f"ALTER SCHEMA {ns} SET TAGS ('domain' = 'customer_experience', 'data_class' = 'synthetic')", tolerate=True)

run(f"""CREATE OR REPLACE FUNCTION {ns}.mask_pii(v STRING)
RETURNS STRING
COMMENT 'Returns the value for members of valtera_voc_pii_readers, otherwise a masked value.'
RETURN CASE WHEN is_account_group_member('valtera_voc_pii_readers') THEN v
            WHEN v IS NULL THEN NULL
            ELSE concat(left(v, 1), repeat('*', 6)) END""")

run(f"""CREATE OR REPLACE VIEW {ns}.customers_secure
COMMENT 'Customer profile with PII masked unless the reader is in valtera_voc_pii_readers. Use this, not silver_customers.'
AS SELECT customer_id, {ns}.mask_pii(first_name) AS first_name, {ns}.mask_pii(last_name) AS last_name,
          {ns}.mask_pii(email) AS email, {ns}.mask_pii(phone) AS phone,
          state, region, loyalty_tier, updated_at
   FROM {ns}.silver_customers""")
print("\n".join(("OK  " if l["ok"] else "SKIP") + "  " + l["sql"][:110] for l in log))

SKIP  ALTER TABLE daria_krasavina.voc_valtera.silver_customers ALTER COLUMN first_name SET TAGS ('pii' = 'first_name
SKIP  ALTER TABLE daria_krasavina.voc_valtera.silver_customers ALTER COLUMN last_name SET TAGS ('pii' = 'last_name')
OK    ALTER TABLE daria_krasavina.voc_valtera.silver_customers ALTER COLUMN email SET TAGS ('pii' = 'email')
SKIP  ALTER TABLE daria_krasavina.voc_valtera.silver_customers ALTER COLUMN phone SET TAGS ('pii' = 'phone')
SKIP  ALTER SCHEMA daria_krasavina.voc_valtera SET TAGS ('domain' = 'customer_experience', 'data_class' = 'synthetic
OK    CREATE OR REPLACE FUNCTION daria_krasavina.voc_valtera.mask_pii(v STRING)
RETURNS STRING
COMMENT 'Returns the 
OK    CREATE OR REPLACE VIEW daria_krasavina.voc_valtera.customers_secure
COMMENT 'Customer profile with PII masked 


In [ ]:
display(spark.sql(f"SELECT * FROM {ns}.customers_secure LIMIT 5"))
tags = spark.sql(f"""SELECT table_name, column_name, tag_name, tag_value
                     FROM {cat}.information_schema.column_tags
                     WHERE schema_name = '{sch}' ORDER BY 1, 2""").toPandas()
print(tags.to_string(index=False))
os.makedirs(ev, exist_ok=True)
with open(f"{ev}/01_governance.json", "w") as f:
    json.dump({"statements": log, "column_tags": tags.to_dict("records"),
               "masked_sample": spark.sql(f"SELECT customer_id, first_name, email, loyalty_tier FROM {ns}.customers_secure LIMIT 3").toPandas().to_dict("records")}, f, indent=2, default=str)

customer_id | first_name | last_name | email | phone | state | region | loyalty_tier | updated_at
C00001 | K****** | H****** | k****** | +****** | CO | West | Standard | 2026-08-01T00:00:00.000Z
C00002 | M****** | L****** | m****** | None | AZ | West | Standard | 2026-08-01T00:00:00.000Z
C00003 | P****** | V****** | p****** | +****** | FL | South | Standard | 2026-08-01T00:00:00.000Z
C00004 | R****** | L****** | r****** | +****** | TX | South | Standard | 2026-08-01T00:00:00.000Z
C00005 | D****** | K****** | d****** | None | NJ | Northeast | Standard | 2026-08-01T00:00:00.000Z
      table_name column_name tag_name tag_value
silver_customers       email      pii     email
